# Synthetic Dataset Generator
Generates realistic synthetic datasets from a schema description using Qwen.

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from huggingface_hub import login, InferenceClient

In [ ]:
load_dotenv()

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct"
client = InferenceClient(MODEL_ID, token=HF_TOKEN)


In [ ]:
# Placeholder for the LLM call logic

def generate_data(user_request, row_count, schema_example, file_type):
    system_prompt = """
    You are an expert Data Engineer specializing in synthetic data generation.
    Your goal is to produce a realistic, high-fidelity dataset that mimics
    real-world distributions and maintains logical consistency across all columns.
    """

    user_prompt = f"""
    User Request: {user_request}

    Requirements:
    - Generate exactly {row_count} rows.
    - Format the output as a valid {file_type} file.
    - Follow this schema structure: {schema_example}

    Only return the raw data, no conversational filler.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
    )
    data = response.choices[0].message.content

    return data

In [ ]:
def create_ui():

    with gr.Blocks(title="Universal Synthetic Data Gen") as demo:
        gr.Markdown("# 🤖 Any-Data Generator")

        with gr.Row():
            with gr.Column():
                user_prompt = gr.Textbox(label="What kind of data do you need?",
                                         placeholder="e.g. Spaceship maintenance logs...", lines=10)
                gr.Examples(["Spaceship maintenance logs"], inputs=[user_prompt])
            with gr.Column():
                schema_example = gr.Textbox(label="Schema Example", lines=10,
                                            placeholder="e.g. start_time, end_time, parts_used, remarks, custodian")
                gr.Examples(
                    ["start_time, end_time, parts_used, remarks, custodian"],
                    inputs=[schema_example]
                )
        with gr.Row():
            row_count = gr.Slider(minimum=1, maximum=100, value=10, step=1, label="Number of Rows")
            file_type = gr.Dropdown(choices=["JSON", "CSV"], value="CSV", label="File Type")
            generate_data_btn = gr.Button("Generate Data", variant="primary")

        output_preview = gr.Textbox(label="Preview")

        generate_data_btn.click(
            fn=generate_data,
            inputs=[user_prompt, row_count, schema_example, file_type],
            outputs=output_preview
        )

    return demo


create_ui().launch()